# Phase 2: Data Quality & Schema Analysis (Member B)

## Tasks B.1-B.3: Quality Metrics & Field Detection

This notebook reproduces the 2014 paper's analysis on:
- **B.1**: Null rate analysis (Figure 13)
- **B.2**: Field type detection (Figure 12)
- **B.3**: Attribute informativeness (Figure 14)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
from urllib.request import urlopen
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the JSON data
with open('../../data/nyc_socrata_datasets.json', 'r') as f:
    data = json.load(f)

print(f'Loaded {len(data)} datasets')
print(f'Sample dataset: {data[0]["name"]}')
print(f'Number of columns: {len(data[0]["columns"])}')

## Task B.1: Null Rate Analysis

Analyze the proportion of null values in each field across all datasets.
Compare with 2014 paper Figure 13 (Table sparseness distribution).

In [ ]:
# Extract null rate information from cachedContents
null_rates = []

for dataset in data:
    full_meta = dataset.get('full_metadata')
    if not full_meta:
        continue
    
    col_details = full_meta.get('column_details') or []
    
    for col in col_details:
        if not col or not isinstance(col, dict):
            continue
        
        cached = col.get('cachedContents') or {}
        
        # Extract null and non-null counts
        try:
            null_count = int(cached.get('null', 0))
            non_null_count = int(cached.get('non_null', 0))
            total = null_count + non_null_count
            
            if total > 0:
                null_rate = null_count / total
            else:
                null_rate = 0
            
            null_rates.append({
                'dataset_name': dataset.get('name'),
                'dataset_id': dataset.get('id'),
                'column_name': col.get('name'),
                'column_type': col.get('type'),
                'null_count': null_count,
                'non_null_count': non_null_count,
                'total_rows': total,
                'null_rate': null_rate
            })
        except (ValueError, TypeError):
            continue

df_null_rates = pd.DataFrame(null_rates)

print(f'Total fields analyzed: {len(df_null_rates)}')
print(f'\nNull rate statistics:')
print(df_null_rates['null_rate'].describe())

# Categorize sparseness (2014 paper uses these bins)
bins = [0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
labels = ['0-10%', '10-20%', '20-30%', '30-50%', '50-70%', '70-100%']
df_null_rates['sparseness_category'] = pd.cut(df_null_rates['null_rate'], bins=bins, labels=labels, include_lowest=True)

print(f'\nSparseness distribution (null rate categories):')
sparseness_counts = df_null_rates['sparseness_category'].value_counts().sort_index()
sparseness_pct = sparseness_counts / len(df_null_rates) * 100
for cat, count in sparseness_counts.items():
    print(f"  {cat}: {count:5d} ({sparseness_pct[cat]:5.1f}%)")

In [ ]:
# Visualization for B.1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of null rates
axes[0].hist(df_null_rates['null_rate'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Null Rate', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Fields', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Field Null Rates', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Sparseness categories (2014 paper comparison)
sparseness_data = df_null_rates['sparseness_category'].value_counts().sort_index()
colors = ['#66B2FF', '#FFB366', '#FF9999', '#FFD699', '#B3D9FF', '#FF9999']
bars = axes[1].bar(range(len(sparseness_data)), sparseness_data.values, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(len(sparseness_data)))
axes[1].set_xticklabels(sparseness_data.index, rotation=45)
axes[1].set_ylabel('Number of Fields', fontsize=12, fontweight='bold')
axes[1].set_title('Field Sparseness Categories', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{int(height)}',
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../../figures/3_1_null_rates.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'2014 Paper Baseline (Figure 13):')
print(f'  Table sparseness 0-0.1 (low null): 63%')
print(f'\n2026 Data:')
print(f'  Fields with 0-10% null: {sparseness_pct["0-10%"]:.1f}%')

## Task B.2: Field Type Detection

Detect special field types: latitude/longitude, date/time, address, zipcode.
Use regex patterns on column names and sample values.
Compare with 2014 paper Figure 12.

In [ ]:
# Field type detection patterns
patterns = {
    'latitude': r'\b(lat|latitude|y_coord)\b',
    'longitude': r'\b(lon|long|longitude|x_coord)\b',
    'date': r'\b(date|created|updated|year|month|day|time|timestamp)\b',
    'address': r'\b(address|street|avenue|road|boulevard|place|location)\b',
    'zipcode': r'\b(zip|postal|postcode|zip_code|postal_code)\b'
}

# Value-based patterns (sample from sample_rows)
def is_coordinate(value):
    """Check if value looks like a coordinate (float between -180 and 180)"""
    try:
        val = float(value)
        return -180 <= val <= 180
    except:
        return False

def is_zipcode(value):
    """Check if value looks like a zipcode (5 digits)"""
    return isinstance(value, str) and re.match(r'^\d{5}(-\d{4})?$', str(value).strip()) is not None

def is_date(value):
    """Check if value looks like a date"""
    if not isinstance(value, str):
        return False
    # Simple check for common date formats
    return bool(re.search(r'\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|\d{4}', str(value)))

# Scan all datasets for field types
field_types = defaultdict(lambda: {'count': 0, 'datasets': set()})
total_fields = 0
datasets_with_geo = 0
datasets_with_date = 0

for dataset in data:
    columns = dataset.get('columns', [])
    sample_rows = dataset.get('sample_rows', [])
    col_details = dataset.get('full_metadata', {}).get('column_details', [])
    
    has_geo = False
    has_date = False
    
    for i, col_name in enumerate(columns):
        total_fields += 1
        col_lower = col_name.lower()
        
        # Check name-based patterns
        for type_name, pattern in patterns.items():
            if re.search(pattern, col_lower):
                field_types[type_name]['count'] += 1
                field_types[type_name]['datasets'].add(dataset.get('id'))
                
                if 'lat' in type_name or 'lon' in type_name:
                    has_geo = True
                if 'date' in type_name:
                    has_date = True
                break
        
        # Check value-based patterns from sample rows
        if sample_rows and i < len(sample_rows[0]):
            # Sample some values from this column
            col_values = [row.get(col_name, '') for row in sample_rows[:20]]
            
            # Check for coordinates
            if any(is_coordinate(v) for v in col_values):
                field_types['coordinate']['count'] += 1
                field_types['coordinate']['datasets'].add(dataset.get('id'))
                has_geo = True
            
            # Check for zipcode
            if any(is_zipcode(v) for v in col_values):
                field_types['zipcode']['count'] += 1
                field_types['zipcode']['datasets'].add(dataset.get('id'))
    
    if has_geo:
        datasets_with_geo += 1
    if has_date:
        datasets_with_date += 1

print(f'Total fields scanned: {total_fields}')
print(f'\nField types detected:')
for type_name, info in sorted(field_types.items(), key=lambda x: x[1]['count'], reverse=True):
    count = info['count']
    datasets = len(info['datasets'])
    pct = count / total_fields * 100 if total_fields > 0 else 0
    print(f"  {type_name:15s}: {count:5d} fields ({pct:5.2f}%) in {datasets:4d} datasets")

print(f'\nDatasets with geolocation: {datasets_with_geo} ({datasets_with_geo/len(data)*100:.1f}%)')
print(f'Datasets with date info: {datasets_with_date} ({datasets_with_date/len(data)*100:.1f}%)')

print(f'\n2014 Paper Baseline (Figure 12):')
print(f'  Lat/Lon presence (all cities): 52.9%')
print(f'  Tables with date info: 40.4%')

In [ ]:
# Visualization for B.2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Field types distribution
type_names = list(field_types.keys())
type_counts = [field_types[t]['count'] for t in type_names]
type_datasets = [len(field_types[t]['datasets']) for t in type_names]

sorted_idx = sorted(range(len(type_counts)), key=lambda i: type_counts[i], reverse=True)
type_names = [type_names[i] for i in sorted_idx]
type_counts = [type_counts[i] for i in sorted_idx]
type_datasets = [type_datasets[i] for i in sorted_idx]

axes[0].barh(type_names, type_counts, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Fields', fontsize=12, fontweight='bold')
axes[0].set_title('Field Type Detection (B.2)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Add count labels
for i, (name, count) in enumerate(zip(type_names, type_counts)):
    axes[0].text(count, i, f' {count}', va='center', fontweight='bold')

# Datasets with special features
feature_names = ['Geolocation\n(Lat/Lon)', 'Date/Time\nInfo', 'Address\nFields', 'Zipcode\nFields']
feature_datasets = [
    datasets_with_geo,
    datasets_with_date,
    len(field_types.get('address', {}).get('datasets', set())),
    len(field_types.get('zipcode', {}).get('datasets', set()))
]
feature_pct = [d/len(data)*100 for d in feature_datasets]

bars = axes[1].bar(feature_names, feature_pct, color=['#FF9999', '#66B2FF', '#FFB366', '#B3FF99'], 
                   edgecolor='black', alpha=0.8)
axes[1].set_ylabel('Percentage of Datasets (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Datasets with Special Field Types', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, max(feature_pct) * 1.2)
axes[1].grid(True, alpha=0.3, axis='y')

# Add percentage labels
for bar, pct in zip(bars, feature_pct):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{pct:.1f}%',
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../../figures/3_2_field_types.png', dpi=300, bbox_inches='tight')
plt.show()

## Task B.3: Attribute Informativeness

Measure field name quality using the Wordlist Dictionary.
Informativeness = proportion of fields with meaningful names.
Compare with 2014 paper Figure 14.

In [ ]:
# Create a sample English dictionary (full Wordlist Dictionary is huge)
# Using common English words for informativeness check
sample_wordlist = {
    'address', 'age', 'amount', 'area', 'attribute', 'borough', 'building', 'business',
    'category', 'census', 'city', 'class', 'code', 'college', 'column', 'complaint',
    'count', 'county', 'created', 'crime', 'data', 'date', 'day', 'department',
    'description', 'district', 'division', 'document', 'domain', 'download', 'drive',
    'education', 'email', 'enabled', 'end', 'event', 'factor', 'fee', 'field',
    'file', 'flag', 'floor', 'format', 'frequency', 'function', 'fund', 'grade',
    'group', 'hash', 'health', 'height', 'historic', 'history', 'holiday', 'home',
    'hospital', 'house', 'housing', 'id', 'identifier', 'image', 'income', 'info',
    'information', 'initial', 'input', 'inspection', 'institution', 'interest', 'issue',
    'item', 'jurisdiction', 'key', 'kind', 'label', 'land', 'language', 'latitude',
    'law', 'layer', 'length', 'level', 'license', 'line', 'link', 'list', 'location',
    'longitude', 'lot', 'lower', 'maintenance', 'manager', 'map', 'market', 'material',
    'maximum', 'measure', 'media', 'meeting', 'member', 'message', 'method', 'metric',
    'middle', 'minimum', 'minute', 'mode', 'model', 'modification', 'month', 'morning',
    'name', 'nature', 'neighborhood', 'network', 'new', 'news', 'node', 'note',
    'number', 'object', 'observation', 'office', 'official', 'old', 'operation', 'operator',
    'option', 'order', 'organization', 'origin', 'output', 'owner', 'page', 'park',
    'part', 'participant', 'party', 'password', 'path', 'pattern', 'payment', 'penalty',
    'people', 'percent', 'performance', 'period', 'permission', 'person', 'phone',
    'photo', 'phrase', 'place', 'plan', 'plate', 'platform', 'point', 'policy',
    'population', 'position', 'postal', 'pound', 'practice', 'preference', 'presence',
    'presentation', 'price', 'primary', 'priority', 'private', 'problem', 'procedure',
    'process', 'producer', 'product', 'profile', 'program', 'project', 'property',
    'proportion', 'proposal', 'protection', 'protocol', 'provider', 'publication',
    'purpose', 'quantity', 'query', 'question', 'range', 'rank', 'rate', 'rating',
    'ratio', 'read', 'reason', 'receipt', 'record', 'recreation', 'reference', 'region',
    'register', 'regulation', 'relation', 'relationship', 'release', 'relevant', 'report',
    'representation', 'request', 'requirement', 'research', 'reservation', 'resident',
    'resolution', 'resource', 'response', 'result', 'return', 'review', 'revision',
    'right', 'road', 'role', 'route', 'rule', 'running', 'safety', 'sale', 'sample',
    'satisfaction', 'scale', 'scene', 'scheme', 'school', 'scope', 'score', 'search',
    'season', 'seat', 'secondary', 'section', 'security', 'segment', 'selection', 'sender',
    'senior', 'sense', 'sensor', 'sentence', 'sentiment', 'sequence', 'serial', 'series',
    'service', 'session', 'setting', 'settlement', 'shape', 'share', 'shelter', 'shift',
    'shop', 'short', 'shot', 'shoulder', 'show', 'signal', 'signature', 'significance',
    'silence', 'similarity', 'simple', 'simulation', 'site', 'situation', 'size', 'skill',
    'social', 'society', 'software', 'solution', 'source', 'space', 'spare', 'spatial',
    'speaker', 'special', 'specification', 'speed', 'spending', 'sphere', 'spider', 'spirit',
    'split', 'sponsor', 'sport', 'spot', 'spread', 'spring', 'staff', 'stage', 'standard',
    'start', 'state', 'statement', 'station', 'status', 'statute', 'step', 'stop',
    'storage', 'store', 'storm', 'story', 'street', 'structure', 'student', 'study',
    'style', 'subject', 'submission', 'subscribe', 'substance', 'substantial', 'substitute',
    'success', 'sufficient', 'suggestion', 'summary', 'summer', 'supply', 'support',
    'surface', 'survey', 'survival', 'suspect', 'suspension', 'system', 'table', 'tag',
    'target', 'task', 'tax', 'teach', 'team', 'technology', 'telephone', 'template',
    'temporary', 'tenant', 'tendency', 'terminal', 'territory', 'test', 'text', 'theme',
    'theory', 'therapy', 'there', 'therefore', 'thermal', 'thick', 'thing', 'thinking',
    'third', 'thorough', 'thought', 'thousand', 'threat', 'threshold', 'throat', 'through',
    'ticket', 'time', 'timeline', 'timestamp', 'timing', 'tip', 'tire', 'title', 'today',
    'tolerance', 'tone', 'tool', 'topic', 'total', 'touch', 'tourist', 'toward', 'tower',
    'town', 'track', 'trade', 'tradition', 'traffic', 'trail', 'train', 'training',
    'transaction', 'transfer', 'transformation', 'transition', 'translation', 'transmission',
    'transparent', 'transport', 'travel', 'treatment', 'tree', 'trend', 'trial', 'triangle',
    'trip', 'trouble', 'truck', 'trust', 'truth', 'tuesday', 'type', 'typical', 'ultimate',
    'umbrella', 'unable', 'unauthorized', 'uncertain', 'uncle', 'uncommon', 'underground',
    'understand', 'undertaking', 'unemployment', 'unexpected', 'uniform', 'union', 'unique',
    'unit', 'united', 'universal', 'universe', 'university', 'unknown', 'unlawful',
    'unnecessary', 'unprecedented', 'unpredictable', 'unreasonable', 'unrelated', 'unsafe',
    'unsuccessful', 'unsustainable', 'unused', 'update', 'updated', 'upgrade', 'upheaval',
    'upper', 'upset', 'urban', 'urged', 'usage', 'use', 'used', 'useful', 'user',
    'username', 'usual', 'usually', 'utility', 'vacant', 'vacation', 'vaccine', 'valid',
    'validity', 'valley', 'value', 'valve', 'variable', 'variation', 'variety', 'various',
    'vehicle', 'vendor', 'venture', 'venue', 'verbal', 'verdict', 'verification', 'verify',
    'version', 'versus', 'vertical', 'vessel', 'veteran', 'viable', 'vibrant', 'vice',
    'victim', 'victory', 'video', 'view', 'violation', 'violence', 'viral', 'virtual',
    'virtue', 'virus', 'visibility', 'visible', 'vision', 'visit', 'visual', 'vital',
    'vocabulary', 'vocal', 'voice', 'volume', 'volunteer', 'vote', 'vulnerable', 'wage',
    'wait', 'walk', 'wall', 'war', 'ward', 'warehouse', 'warning', 'warrant', 'waste',
    'water', 'wave', 'way', 'wealth', 'weapon', 'wear', 'weather', 'website', 'wednesday',
    'week', 'weekend', 'weight', 'welcome', 'welfare', 'western', 'wet', 'what', 'whatever',
    'wheat', 'wheel', 'when', 'whenever', 'where', 'wherever', 'whether', 'which',
    'while', 'whisper', 'white', 'whole', 'wholesale', 'whom', 'whose', 'wicked', 'wide',
    'widely', 'width', 'wife', 'wild', 'wildlife', 'will', 'willing', 'window', 'wine',
    'wing', 'winner', 'winter', 'wire', 'wireless', 'wisdom', 'wish', 'witness', 'woman',
    'wonder', 'wonderful', 'wood', 'word', 'work', 'worker', 'workplace', 'world', 'worried',
    'worry', 'worse', 'worship', 'worst', 'worth', 'worthy', 'would', 'wound', 'wrap',
    'write', 'writer', 'writing', 'written', 'wrong', 'yard', 'year', 'yell', 'yellow',
    'yesterday', 'yield', 'york', 'young', 'younger', 'youngest', 'your', 'yours', 'yourself',
    'youth', 'zone', 'zipcode', 'zip', 'zone', 'zoom'
}

print(f'Loaded {len(sample_wordlist)} words for informativeness check')
print(f'Note: Using sample wordlist. Full Wordlist Dictionary has ~170k words')

In [ ]:
# Calculate attribute informativeness for each dataset
informativeness_scores = []

for dataset in data:
    columns = dataset.get('columns', [])
    
    if not columns:
        continue
    
    informative_count = 0
    
    for col_name in columns:
        # Split column name by underscore, space, or camelCase
        tokens = re.split(r'[_\s]+|(?<=[a-z])(?=[A-Z])', col_name.lower())
        tokens = [t.strip() for t in tokens if len(t.strip()) > 2]  # Filter tokens > 2 chars
        
        # Check if all tokens are in wordlist
        if tokens and all(t in sample_wordlist for t in tokens):
            informative_count += 1
    
    informativeness = informative_count / len(columns) if columns else 0
    
    informativeness_scores.append({
        'dataset_id': dataset.get('id'),
        'dataset_name': dataset.get('name'),
        'num_columns': len(columns),
        'informative_columns': informative_count,
        'informativeness': informativeness
    })

df_informativeness = pd.DataFrame(informativeness_scores)

print(f'Attribute Informativeness Statistics:')
print(df_informativeness['informativeness'].describe())

# Categorize by informativeness levels
def categorize_informativeness(score):
    if score >= 0.8:
        return 'High (>= 80%)'
    elif score >= 0.6:
        return 'Medium (60-80%)'
    elif score >= 0.4:
        return 'Low (40-60%)'
    else:
        return 'Very Low (< 40%)'

df_informativeness['category'] = df_informativeness['informativeness'].apply(categorize_informativeness)

print(f'\nInformativeness Distribution:')
for cat in ['High (>= 80%)', 'Medium (60-80%)', 'Low (40-60%)', 'Very Low (< 40%)']:
    count = (df_informativeness['category'] == cat).sum()
    pct = count / len(df_informativeness) * 100
    print(f"  {cat:25s}: {count:5d} ({pct:5.1f}%)")

In [ ]:
# Visualization for B.3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of informativeness scores
axes[0].hist(df_informativeness['informativeness'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df_informativeness['informativeness'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {df_informativeness["informativeness"].mean():.2f}')
axes[0].axvline(df_informativeness['informativeness'].median(), color='green', linestyle='--', 
                linewidth=2, label=f'Median: {df_informativeness["informativeness"].median():.2f}')
axes[0].set_xlabel('Attribute Informativeness Score', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Datasets', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Attribute Informativeness', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Category distribution
cat_counts = df_informativeness['category'].value_counts()
cat_order = ['High (>= 80%)', 'Medium (60-80%)', 'Low (40-60%)', 'Very Low (< 40%)']
cat_counts = cat_counts.reindex([c for c in cat_order if c in cat_counts.index])

colors = ['#66B2FF', '#FFB366', '#FF9999', '#CCCCCC']
bars = axes[1].bar(range(len(cat_counts)), cat_counts.values, color=colors[:len(cat_counts)], 
                   edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(len(cat_counts)))
axes[1].set_xticklabels(cat_counts.index, rotation=15, ha='right')
axes[1].set_ylabel('Number of Datasets', fontsize=12, fontweight='bold')
axes[1].set_title('Datasets by Informativeness Category', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{int(height)}',
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../../figures/3_3_informativeness.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary: B.1-B.3 Results

### Key Findings:

**B.1 - Null Rate Analysis:**
- Analyzed sparseness (null rate distribution) across all fields
- Compared with 2014 paper Figure 13

**B.2 - Field Type Detection:**
- Detected geolocation, date, address, and zipcode fields
- Found presence of special field types in datasets
- Compared with 2014 paper Figure 12

**B.3 - Attribute Informativeness:**
- Measured field naming quality using English word dictionary
- Calculated informativeness scores for each dataset
- Compared with 2014 paper Figure 14